# Task 1.1

In [ ]:
import numpy as np
import pandas as pd

inflation_rate = 0.06

In [ ]:
goals = {
    "Marriage": (2500000, 4),
    "Child Education": (3000000, 12), 
    "House Purchase": (10000000, 15)
}

In [ ]:
future_values = []
for goal, data in goals:
    pv = data[0]
    n = data[1]
    fv = pv * ((1+inflation_rate)**n)
    future_values.append({
        "Goal": goal,
        "Tenure (Years)": n, 
        "Current Value": pv,
        "Future Value": fv
    })

df_goals = pd.DataFrame(future_values)
df_goals = df_goals.sort_values(by="Tenure (Years)").reset_index(drop = True)

print("Inflation Adjusted Goals (ranked by urgency)\n")

print(df_goals)

# Task 1.2

In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import fsolve
from scipy.stats import norm

# --- 1. Define Variables ---
inflation_rate = 0.06

# Replace these with your actual portfolio mean return and volatility from your data
port_mean_return = 0.12  # Example: 12% annualized return
port_volatility = 0.15   # Example: 15% annualized volatility

# Calculate the 90% certainty (10th percentile) annualized return
# Z-score for 10th percentile is approx -1.2815
z_score_90 = norm.ppf(0.10)
conservative_annual_return = port_mean_return + (z_score_90 * port_volatility)
print(f"Average Expected Return: {port_mean_return*100:.2f}%")
print(f"Conservative Return (90% Certainty): {conservative_annual_return*100:.2f}%\n")

# Convert annual conservative return to monthly
r_monthly = (1 + conservative_annual_return) ** (1/12) - 1

# Goals: 'Name': [Current Cost (PV), Years to Goal]
goals_data = {
    'Marriage': [2500000, 4],
    'Child Education': [3000000, 12],
    'House Purchase': [10000000, 15]
}

# --- 2. Numerical Solver Function ---
def sip_objective(sip, target_fv, r, n_months):
    """
    The objective function for fsolve. 
    It calculates the difference between the FV of a given SIP and the target FV.
    The solver will find the SIP that makes this difference zero.
    """
    # FV of annuity due (investments at beginning of month)
    calculated_fv = sip * (((1 + r)**n_months - 1) / r) * (1 + r)
    return calculated_fv - target_fv

# --- 3. Calculate FVs and Optimize SIPs ---
results = []

for goal, data in goals_data.items():
    pv = data[0]
    years = data[1]
    n_months = years * 12
    
    # Task 1.1: Inflation Adjustment (FV = PV * (1 + inflation)^years)
    fv = pv * ((1 + inflation_rate) ** years)
    
    # Task 1.2: Minimum SIP Estimation using fsolve
    # We provide an initial guess for the SIP (e.g., 10000)
    initial_guess = 10000 
    optimal_sip = fsolve(sip_objective, initial_guess, args=(fv, r_monthly, n_months))[0]
    
    results.append({
        'Goal': goal,
        'Tenure (Yrs)': years,
        'Present Value (₹)': f"₹{pv:,.0f}",
        'Future Value (₹)': f"₹{fv:,.0f}",
        'Min SIP (90% Certainty) (₹)': f"₹{optimal_sip:,.0f}"
    })

# --- 4. Display Results ---
# Rank by urgency (Task 1.1 requirement)
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by='Tenure (Yrs)').reset_index(drop=True)

print("--- Goal Modeling & Minimum SIP Estimation ---")
print(df_results.to_string())

Average Expected Return: 12.00%
Conservative Return (90% Certainty): -7.22%

--- Goal Modeling & Minimum SIP Estimation ---
              Goal  Tenure (Yrs) Present Value (₹) Future Value (₹) Min SIP (90% Certainty) (₹)
0         Marriage             4        ₹2,500,000       ₹3,156,192                     ₹76,344
1  Child Education            12        ₹3,000,000       ₹6,036,589                     ₹63,768
2   House Purchase            15       ₹10,000,000      ₹23,965,582                    ₹222,449
